In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm

# CONFIGURATION
CANCER = "cancer"
NORMAL= "normal"
PREPARED_DIR = Path("../.data/prepared")
FEATURES_DIR = Path("../.data/features")
WITHOUT_LABEL="sans_label"
WITH_LABEL="avec_labels"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32

# TRANSFORMATIONS
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# SIMPLE DATASET
class BrainMRIDataset(Dataset):
    def __init__(self, image_folder, transform=None):
        self.image_paths = list(image_folder.glob("*.png")) + list(image_folder.glob("*.jpg"))
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        return image, str(img_path.name)


# CHARGER RESNET PRE ENTRAINE
resnet = models.resnet50(pretrained=True)
# On retire la dernière couche (classification)
resnet = nn.Sequential(*list(resnet.children())[:-1])
resnet = resnet.to(DEVICE)
resnet.eval()  # Mode évaluation (pas d'entraînement)

print(f"ResNet50 chargé sur {DEVICE}")

# EXTRACTION DE FEATURES
def extract_features(dataset, model, batch_size=BATCH_SIZE):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    features_list = []
    names_list = []
    
    with torch.no_grad():  # Pas de calcul de gradient
        for images, names in tqdm(loader, desc="Extraction features"):
            images = images.to(DEVICE)
            features = model(images)
            features = features.squeeze()  # Enlever dimensions inutiles
            
            features_list.append(features.cpu().numpy())
            names_list.extend(names)
    
    # Concaténer tous les batches
    all_features = np.vstack(features_list)
    
    return all_features, names_list

# Cancer
dataset_cancer = BrainMRIDataset(PREPARED_DIR / WITH_LABEL / CANCER, transform=transform)
features_cancer, names_cancer = extract_features(dataset_cancer, resnet)
print(f"Cancer: {features_cancer.shape}")

# Normal
dataset_normal = BrainMRIDataset(PREPARED_DIR / WITH_LABEL / NORMAL, transform=transform)
features_normal, names_normal = extract_features(dataset_normal, resnet)
print(f"Normal: {features_normal.shape}")

# Sans labels
dataset_unlabeled = BrainMRIDataset(PREPARED_DIR / WITHOUT_LABEL, transform=transform)
features_unlabeled, names_unlabeled = extract_features(dataset_unlabeled, resnet)
print(f"Sans labels: {features_unlabeled.shape}")

# SAUVEGARDE DES FEATURES
np.save(FEATURES_DIR / "features_cancer.npy", features_cancer)
np.save(FEATURES_DIR / "features_normal.npy", features_normal)
np.save(FEATURES_DIR / "features_unlabeled.npy", features_unlabeled)

print("Features extraites et sauvegardées !")

ResNet50 chargé sur cpu


Extraction features: 100% 2/2 [00:01<00:00,  1.18it/s]


Cancer: (50, 2048)


Extraction features: 100% 2/2 [00:00<00:00, -3.18it/s]


Normal: (50, 2048)


Extraction features: 100% 44/44 [00:47<00:00,  1.08s/it]

Sans labels: (1406, 2048)
Features extraites et sauvegardées !
